# Notebook Setup

## Setting Directories

In [1]:
import sys
import os
from pathlib import Path

# Path to your project root
project_dir = Path(r"C:\Users\dmika\DEV\Projects-local\dp100-learn")

# Change the working directory
os.chdir(project_dir)

# Add to sys.path if not already there
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

## Imports

### Local Imports

In [2]:
# Now you can import from utils
from utils.consts import SUBSCRIPTION_ID, PREFERED_RESOURCE_LOCATION, MAIN_STORAGE_ACCOUNT_ACCESS_KEY

### General Imports

In [3]:
import numpy as np
import pandas as pd

### Azure Imports

In [4]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

## Consts

In [5]:
subscription_id = SUBSCRIPTION_ID
azure_credentials = DefaultAzureCredential()

## Other

In [6]:
resource_group_name = "ml-workspace-dev"
resource_group_location = PREFERED_RESOURCE_LOCATION

In [7]:
storage_account_name = "dmpdp100storageaccount99"  # must be globally unique
storage_account_location = PREFERED_RESOURCE_LOCATION
storage_container_name = "dmpdp100data"
storage_account_access_key = MAIN_STORAGE_ACCOUNT_ACCESS_KEY

In [8]:
azureml_workspace_name = "mlw-dp100-labs"
azureml_resource_location = PREFERED_RESOURCE_LOCATION

In [9]:
datastore_name = "dmdp100datastore"

In [10]:
ml_client = MLClient(
    credential=azure_credentials,
    subscription_id=subscription_id,
    resource_group_name=resource_group_name,
    workspace_name=azureml_workspace_name
)

# Setup a Resource Group

In [11]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.resource import ResourceManagementClient

# Create resource management client
resource_client = ResourceManagementClient(azure_credentials, subscription_id)
try:
    # Try to get existing resource group
    rg_result = resource_client.resource_groups.get(resource_group_name)
    print(f"Resource group '{rg_result.name}' already exists in region '{rg_result.location}'")
except ResourceNotFoundError:
    # Create resource group if it doesn't exist
    rg_result = resource_client.resource_groups.create_or_update(
        resource_group_name,
        {"location": resource_group_location}
    )
    print(f"Provisioned resource group '{rg_result.name}' in the {rg_result.location} region")


# Optional lines to delete the resource group. begin_delete is asynchronous.
# poller = resource_client.resource_groups.begin_delete(rg_result.name)
# result = poller.result()

Resource group 'ml-workspace-dev' already exists in region 'westeurope'


# Setup Storage Account

## Create a Storage Account

In [12]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.storage import StorageManagementClient

storage_client = StorageManagementClient(azure_credentials, subscription_id)

try:
    # Try to get existing storage account
    storage_account = storage_client.storage_accounts.get_properties(
        resource_group_name=resource_group_name,
        account_name=storage_account_name
    )
    print(f"Storage account already exists: {storage_account.name}")
except ResourceNotFoundError:
    # If not found, create new storage account
    print("Creating storage account...")
    poller = storage_client.storage_accounts.begin_create(
        resource_group_name=resource_group_name,
        account_name=storage_account_name,
        parameters={
            "location": storage_account_location,
            "sku": {"name": "Standard_LRS"},
            "kind": "StorageV2",
            "enable_https_traffic_only": True
        }
    )
    storage_account = poller.result()
    print(f"Storage account created: {storage_account.name}")

# Extract the resource ID
storage_resource_id = storage_account.id
print(f"Storage Resource ID: {storage_resource_id}")

Storage account already exists: dmpdp100storageaccount99
Storage Resource ID: /subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.Storage/storageAccounts/dmpdp100storageaccount99


## Create Data Container

In [13]:
from azure.storage.blob import BlobServiceClient

# Build connection string
connection_string = (
    f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};"
    f"AccountKey={storage_account_access_key};EndpointSuffix=core.windows.net"
)

# Create blob service client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create container (if not exists)
try:
    blob_service_client.create_container(storage_container_name)
    print(f"Container '{storage_container_name}' created.")
except Exception as e:
    if "ContainerAlreadyExists" in str(e):
        print(f"Container '{storage_container_name}' already exists.")
    else:
        raise

Container 'dmpdp100data' created.


# Setup Azure ML Workspace

## Creating the Workspace

In [14]:
from azure.core.exceptions import ResourceNotFoundError
from azure.ai.ml.entities import Workspace

try:
    # Try to get existing workspace
    ws = ml_client.workspaces.get(azureml_workspace_name)
    print(f"AML workspace already exists: {ws.name}")
except (ResourceNotFoundError, TypeError):
    ml_client = MLClient(
        credential=azure_credentials,
        subscription_id=subscription_id,
        resource_group_name=resource_group_name,
    )
    # Create new AML workspace
    # NOTE: if creating a new workspace fails with workspace name already exists, it might be that ML workspace was not permanently deleted. To do so you need to go to Azure Portal -> Azure Machine Learning and delete it from there.
    ws = Workspace(
        name=azureml_workspace_name,
        location=azureml_resource_location,
        storage_account=storage_resource_id,
    )
    print("Creating AML workspace...")
    ml_client.workspaces.begin_create(ws).result()
    print(
        f"Workspace '{azureml_workspace_name}' created with default storage: {storage_account_name}"
    )
    ml_client = MLClient(
        credential=azure_credentials,
        subscription_id=subscription_id,
        resource_group_name=resource_group_name,
        workspace_name=azureml_workspace_name,
    )

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Creating AML workspace...


The deployment request mlw-dp100-labs-4410751 was accepted. ARM deployment URI for reference: 
https://portal.azure.com//#blade/HubsExtension/DeploymentDetailsBlade/overview/id/%2Fsubscriptions%2Fa1267753-4c98-48c1-a8e9-9c7169202ffd%2FresourceGroups%2Fml-workspace-dev%2Fproviders%2FMicrosoft.Resources%2Fdeployments%2Fmlw-dp100-labs-4410751
Creating Key Vault: (mlwdp100keyvault5e1d7212  ) ..  Done (17s)
Creating Log Analytics Workspace: (mlwdp100logalyti01c90a7f  )   Done (21s)
Creating AzureML Workspace: (mlw-dp100-labs  ) .  Done (17s)
Creating Application Insights: (mlwdp100insights2705a3ff  )  Done (24s)
Total time : 43s



Workspace 'mlw-dp100-labs' created with default storage: dmpdp100storageaccount99


Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## Setup the Data

### Create a DataStore

In [15]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration

store = AzureBlobDatastore(
    name=datastore_name,
    description="Blob Storage for DP-100 certification prep",
    account_name=storage_account_name,
    container_name=storage_container_name, 
    credentials=AccountKeyConfiguration(
        account_key=storage_account_access_key
    ),
    type="azure_blob",
)

ml_client.create_or_update(store)

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'dmdp100datastore', 'description': 'Blob Storage for DP-100 certification prep', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/datastores/dmdp100datastore', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x0000018CF755D7E0>, 'credentials': {'type': 'account_key'}, 'container_name': 'dmpdp100data', 'account_name': 'dmpdp100storageaccount99', 'endpoint': 'core.windows.net', 'protocol': 'https'})

In [16]:
stores = ml_client.datastores.list()
for ds_name in stores:
    print(ds_name.name)

dmdp100datastore
workspaceartifactstore
workspacefilestore
workspaceworkingdirectory
workspaceblobstore


### Create Data Assets

#### URI_FILE

In [17]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/azure-ml-labs-data/diabetes/diabetes.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="diabetes-data-file",
)

ml_client.data.create_or_update(my_data)

Uploading diabetes.csv (< 1 MB): 100%|##########| 518k/518k [00:00<00:00, 2.60MB/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/3d8efe6c7dafbd432031e3f030cc92dd5acd9da03214f6d3ac06b02ed81cc551/diabetes.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data-file', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/diabetes-data-file/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at

In [18]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/telco-churn-data/telco-customer-churn.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="telco-churn-file-raw"
)

ml_client.data.create_or_update(my_data)

Uploading telco-customer-churn.csv (< 1 MB): 100%|##########| 970k/970k [00:00<00:00, 2.78MB/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/c9f74bf3dd2417bba280f0ceef276024cbe95a3935ed06a94a7f357d15495775/telco-customer-churn.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-file-raw', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-file-raw/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.Syst

#### URI_FOLDER

In [19]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_path = './data/telco-churn-data'

my_data = Data(
    path=folder_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FOLDER,
    description="Data asset pointing to data-asset-path folder in datastore",
    name="telco-churn-folder-raw",
)

ml_client.data.create_or_update(my_data)

Uploading telco-churn-data (0.97 MBs): 100%|##########| 970595/970595 [00:00<00:00, 2295933.66it/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/67bbe0434ee4e9154f85f403f18de4e49ef43c66b5844f85418a3218b89d8b33/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-folder-raw', 'description': 'Data asset pointing to data-asset-path folder in datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-folder-raw/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000018C

#### MLTABLE

In [20]:
%%writefile data/azure-ml-labs-data/diabetes/MLTable

paths:
  - file: ./diabetes.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Writing data/azure-ml-labs-data/diabetes/MLTable


In [21]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/azure-ml-labs-data/diabetes'

my_data = Data(
    path=data_path,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to diabetes.csv in data folder",
    name="diabetes-data-table",
    datastore=datastore_name,

)

ml_client.data.create_or_update(my_data)

Uploading diabetes (0.52 MBs): 100%|##########| 517878/517878 [00:00<00:00, 1556235.70it/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/e4dce288fefaf138d80b1b6b947580788e78dde2c70eb2276db86a6381d3569b/diabetes/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./diabetes.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data-table', 'description': 'MLTable pointing to diabetes.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/diabetes-data-table/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000018CF78866E0>, 'se

In [22]:
%%writefile data/telco-churn-data/MLTable

paths:
  - file: ./telco-customer-churn.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Overwriting data/telco-churn-data/MLTable


In [23]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/telco-churn-data'

my_data = Data(
    path=data_path,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to telco-customer-churn.csv in data folder",
    name="telco-churn-table-raw",
    datastore=datastore_name,

)

ml_client.data.create_or_update(my_data)

Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/67bbe0434ee4e9154f85f403f18de4e49ef43c66b5844f85418a3218b89d8b33/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./telco-customer-churn.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-table-raw', 'description': 'MLTable pointing to telco-customer-churn.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-table-raw/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemDat

### Read the Data Assets Locally

In [24]:
data_asset = ml_client.data.get("telco-churn-file-raw", version="1")
df = pd.read_csv(data_asset.path)
df.sample(2)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
6139,3039-MJSLN,Male,0,No,Yes,3,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),20.2,50.6,No
4026,0266-GMEAO,Male,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),114.3,8058.55,No


In [25]:
# import mltable

data_asset = ml_client.data.get("telco-churn-folder-raw", version="1")
path = {
  'folder': data_asset.path
}
# tbl = mltable.from_delimited_files(paths=[path])
# df = tbl.to_pandas_dataframe()
df = pd.read_csv(os.path.join(data_asset.path, 'telco-customer-churn.csv'))
df.sample(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
5562,8023-QHAIO,Female,1,Yes,No,56,Yes,Yes,Fiber optic,No,...,No,Yes,No,No,Month-to-month,Yes,Bank transfer (automatic),76.85,4275.75,No
1814,5442-PPTJY,Male,0,Yes,Yes,12,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.70,258.35,No


In [27]:
import mltable

data_asset = ml_client.data.get("diabetes-data-table", version="1")

tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,False
1,1147438,8,92,93,47,36,21.240576,0.158365,23,False
2,1640031,7,115,47,52,35,41.511523,0.079019,23,False
3,1883350,9,103,78,25,304,29.582192,1.282870,43,True
4,1424119,1,85,59,27,35,42.604536,0.549542,22,False


In [26]:
import mltable

data_asset = ml_client.data.get("telco-churn-table-raw", version="1")

tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,False,True,False,1,False,No phone service,DSL,No,...,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,False
1,5575-GNVDE,Male,False,False,False,34,True,No,DSL,Yes,...,Yes,No,No,No,One year,False,Mailed check,56.95,1889.50,False
2,3668-QPYBK,Male,False,False,False,2,True,No,DSL,Yes,...,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,True
3,7795-CFOCW,Male,False,False,False,45,False,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.30,1840.75,False
4,9237-HQITU,Female,False,False,False,2,True,No,Fiber optic,No,...,No,No,No,No,Month-to-month,True,Electronic check,70.70,151.65,True


## Setup an Environment

In [28]:
env_file_path = "python_env_setup/dmdp100env.yml"
main_env_name = "dmdp100env"
parent_image = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env_content = f"""
channels:
  - conda-forge
dependencies:
  - python=3.10.11
  - pip=22.3.1
  - pip:
      - scipy
      - pandas
      - scikit-learn
      - adlfs
      - fsspec
      - xgboost
      - lightgbm
      - mlflow
      - azureml-mlflow
      - matplotlib
      - tqdm
      - seaborn
name: {main_env_name}
"""

with open(env_file_path, "w") as f:
    f.write(env_content)


In [29]:
from azure.ai.ml.entities import Environment

env_docker_conda = Environment(
    image=parent_image,
    conda_file=env_file_path,
    name=main_env_name,
    description="Environment created for main dp100 prep.",
)
ml_client.environments.create_or_update(env_docker_conda)
# NOTE: TO create an environment image you have to have a premium COntainer Registry and update your workspace accordingly:
# ws.image_build_compute = "your-cluster"
# ml_client.workspaces.begin_update(ws)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'dmdp100env', 'description': 'Environment created for main dp100 prep.', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/environments/dmdp100env/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000018CFA65A890>, 'serialize': <msrest.serialization.Serializer object at 0x0000018CFA69DCC0>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.10.11', 'pip=22.3.1', {'pip': ['s

In [30]:
# Working setup
# mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04

# channels:
#   - conda-forge
# dependencies:
#   - python=3.11
#   - scikit-learn
#   - pandas
#   - numpy
#   - matplotlib
#   - mlflow
# name: dmdp100env


In [31]:
# %%writefile $env_file_path
# name: dmdp100env
# channels:
#   - conda-forge
# dependencies:
#   - python=3.11
#   - scikit-learn
#   - pandas
#   - numpy
#   - matplotlib
#   - pip
#   - pip:
#     - seaborn
#     - mltable
#     - mlflow
#     - azureml-mlflow

In [32]:
# %%writefile $env_file_path
# name: dmdp100env
# channels:
#   - conda-forge
# dependencies:
#   - python=3.11
#   - scikit-learn
#   - pandas
#   - numpy
#   - matplotlib
#   - mlflow
#   - pip
#   - pip:
#     - seaborn

In [33]:
# from azure.ai.ml.entities import Environment

# env_docker_conda = Environment(
#     image="mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04",
#     conda_file=env_file_path,
#     name="dmpdp100env",
#     description="Environment created for main dp100 prep.",
# )
# ml_client.environments.create_or_update(env_docker_conda)

In [34]:
# %%writefile python_env_setup/mlflowenv.yml
# name: mlflowenv
# channels:
#   - conda-forge
# dependencies:
# - python=3.12.3
# - pip<=23.1.2
# - pip:
#   - mlflow
#   - scikit-learn==1.6.1
#   - psutil==5.9.4
#   - pandas
#   - numpy

In [35]:
# from azure.ai.ml.entities import Environment

# env_docker_conda = Environment(
#     image="mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04",
#     conda_file="python_env_setup/mlflowenv.yml",
#     name="mlflowenv",
#     description="Environment created for MLflow.",
# )
# ml_client.environments.create_or_update(env_docker_conda)

## Setup the Compute

### Compute Cluster

In [36]:
from azure.ai.ml.entities import AmlCompute

# Name assigned to the compute cluster
cpu_compute_target = "dmdp100-cpu-cluster"

try:
    # let's see if the compute target already exists
    cpu_cluster = ml_client.compute.get(cpu_compute_target)
    print(
        f"You already have a cluster named {cpu_compute_target}, we'll reuse it as is."
    )

except Exception:
    print("Creating a new cpu compute target...")

    # Let's create the Azure ML compute object with the intended parameters
    cpu_cluster = AmlCompute(
        name=cpu_compute_target,
        # Azure ML Compute is the on-demand VM service
        type="amlcompute",
        # VM Family
        size="STANDARD_DS11_V2",
        # Minimum running nodes when there is no job running
        min_instances=0,
        # Nodes in cluster
        max_instances=1,
        # How many seconds will the node running after the job termination
        idle_time_before_scale_down=120,
        # Dedicated or LowPriority. The latter is cheaper but there is a chance of job termination
        tier="Dedicated",
    )

    # Now, we pass the object to MLClient's create_or_update method
    cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster)


Creating a new cpu compute target...


### Compute Instance

In [ ]:
# Compute Instances need to have a unique name across the region.
# Here we create a unique name with current datetime
from azure.ai.ml.entities import ComputeInstance
import datetime

ci_basic_name = "dp100ci" + datetime.datetime.now().strftime("%Y%m%d%H%M")
ci_basic_name = ci_basic_name[:24]
ci_basic = ComputeInstance(name=ci_basic_name, size="STANDARD_DS11_V2", idle_time_before_shutdown_minutes="15")
ml_client.begin_create_or_update(ci_basic).result()

## Setup Workspace Defaults

In [ ]:
# Name assigned to the compute cluster
cpu_compute_target = "dmdp100-cpu-cluster"

# Update workspace default compute
workspace = ml_client.workspaces.get(azureml_workspace_name)
workspace.default_compute = cpu_compute_target
ml_client.workspaces.begin_update(workspace)

In [23]:
workspace.storage_account

'/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.Storage/storageAccounts/dmpdp100storageaccount99'

In [20]:
main_datastore = ml_client.datastores.get(datastore_name)
ml_client.datastores.set_default(main_datastore.name)

AttributeError: 'DatastoreOperations' object has no attribute 'set_default'

In [14]:
# Name assigned to the compute cluster
cpu_compute_target = "dmdp100-cpu-cluster"

In [25]:
workspace.environments

AttributeError: 'Workspace' object has no attribute 'environments'